# Phase 4 Alternative - Fast ETCCDI Index-First Bias Correction

This notebook is the fast alternative to daily QDM. It avoids grid-cell daily QDM and instead:

1. Computes annual ETCCDI-style indices from raw daily CMIP6.
2. Computes annual ETCCDI-style indices from ERA5-Land for 1985-2014.
3. Bias-corrects the annual CMIP6 index series against ERA5 annual indices.
4. Saves corrected annual index rasters, zone summaries, period-change tables, and plots.

This is much faster and provides the outputs needed for maps, trends, scenario comparisons, and vegetation linkage analysis.

## Cell 1 - Imports and Configuration

In [ ]:
from pathlib import Path
import gc
import re
import warnings

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib as mpl

warnings.filterwarnings('ignore')

ROOT = (Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve())
CMIP_DAILY_ROOT = ROOT / 'output' / 'cmip6_daily'
ERA5_ROOT = ROOT / 'output' / 'era5'
ZONES_FILE = ROOT / 'output' / 'zones' / 'hydroclimatic_zones_SA.nc'

OUT_ROOT = ROOT / 'output' / 'etccdi_fast'
RAW_DIR = OUT_ROOT / 'raw_indices'
ERA5_INDEX_DIR = OUT_ROOT / 'era5_indices'
CORR_DIR = OUT_ROOT / 'corrected_indices'
TABLE_DIR = OUT_ROOT / 'tables'
FIG_DIR = OUT_ROOT / 'figures'
LOG_DIR = OUT_ROOT / 'logs'

for d in [RAW_DIR, ERA5_INDEX_DIR, CORR_DIR, TABLE_DIR, FIG_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

FULL_MODELS = ['CanESM5', 'GFDL-ESM4', 'INM-CM5-0', 'IPSL-CM6A-LR', 'MPI-ESM1-2-HR']
PRECIP_ONLY_MODELS = ['CESM2']
MODELS = FULL_MODELS + PRECIP_ONLY_MODELS

SCENARIOS = {
    'historical': (1985, 2014),
    'ssp245': (2015, 2100),
    'ssp585': (2015, 2100),
}
BASELINE = (1985, 2014)
# Observed ERA5 index record. Extend to 2024 so the MODIS overlap (2001-2025)
# used in the vegetation-climate linkage is ~24 yr instead of ~14 yr.
# Requires ERA5-Land daily pr/tasmax/tasmin files for 2015-2024 in output/era5/<var>/.
OBS_PERIOD = (1985, 2024)

# Use for testing, e.g. (1985, 1986) or (2015, 2016). Set None for full run.
TEST_YEARS = None

# Keep False to resume safely without reprocessing corrected index files already saved.
OVERWRITE_CORRECTED = False
# Rebuild only these count/duration indices if an earlier run stored them as timedeltas.
FORCE_REPROCESS_CORRECTED_INDICES = ['R10mm', 'R20mm', 'CDD', 'CWD', 'SU', 'TR', 'FD', 'ID']

PRECIP_INDICES = ['PRCPTOT', 'RX1day', 'RX5day', 'SDII', 'R10mm', 'R20mm', 'CDD', 'CWD']
TEMP_INDICES = ['TXx', 'TXn', 'TNx', 'TNn', 'DTR', 'SU', 'TR', 'FD', 'ID']
ALL_INDICES = PRECIP_INDICES + TEMP_INDICES

INDEX_UNITS = {
    'PRCPTOT': 'mm', 'RX1day': 'mm', 'RX5day': 'mm', 'SDII': 'mm day-1',
    'R10mm': 'days', 'R20mm': 'days', 'CDD': 'days', 'CWD': 'days',
    'TXx': 'degC', 'TXn': 'degC', 'TNx': 'degC', 'TNn': 'degC', 'DTR': 'degC',
    'SU': 'days', 'TR': 'days', 'FD': 'days', 'ID': 'days',
}

print('Output root:', OUT_ROOT)

## Cell 2 - Helpers

In [ ]:
def standardise_xy(ds):
    rename = {}
    for cand in ['latitude', 'y']:
        if cand in ds.coords or cand in ds.dims:
            rename[cand] = 'lat'
    for cand in ['longitude', 'x']:
        if cand in ds.coords or cand in ds.dims:
            rename[cand] = 'lon'
    if rename:
        ds = ds.rename(rename)
    if 'lon' in ds.coords and float(ds.lon.max()) > 180:
        ds = ds.assign_coords(lon=((ds.lon + 180) % 360) - 180).sortby('lon')
    if 'lat' in ds.coords:
        ds = ds.sortby('lat')
    if 'lon' in ds.coords:
        ds = ds.sortby('lon')
    return ds


def first_data_var(ds):
    return list(ds.data_vars)[0] if ds.data_vars else None


def is_structurally_readable(path):
    if (not path.exists()) or path.stat().st_size < 1_000_000:
        return False
    try:
        with xr.open_dataset(path) as ds:
            return bool(ds.data_vars) and ('time' in ds.dims or 'time' in ds.coords)
    except Exception:
        return False


def cmip_file(model, scenario, variable, year):
    return CMIP_DAILY_ROOT / scenario / model / variable / f'{model}_{variable}_{scenario}_{year}_daily.nc'


def era5_file(variable, year):
    return ERA5_ROOT / variable / f'era5_land_{variable}_{year}_clipped.nc'


def load_daily_file(path, variable):
    ds = xr.open_dataset(path)
    ds = standardise_xy(ds)
    name = variable if variable in ds.data_vars else first_data_var(ds)
    da = ds[name].rename(variable).transpose('time', 'lat', 'lon')
    if variable in ['tasmax', 'tasmin']:
        units = str(da.attrs.get('units', '')).lower()
        # ERA5 files are K; CMIP daily exports are already degC.
        try:
            sample_mean = float(da.isel(time=slice(0, min(10, da.sizes['time']))).mean().values)
        except Exception:
            sample_mean = np.nan
        if units in ['k', 'kelvin'] or sample_mean > 100:
            da = da - 273.15
    da.attrs['units'] = 'mm day-1' if variable == 'pr' else 'degC'
    return da


def years_for(scenario):
    start, end = SCENARIOS[scenario]
    if TEST_YEARS is not None:
        start = max(start, TEST_YEARS[0])
        end = min(end, TEST_YEARS[1])
    return range(start, end + 1)


def max_consecutive_true(mask):
    mask = np.asarray(mask, dtype=bool)
    best = 0
    run = 0
    for val in mask:
        if val:
            run += 1
            best = max(best, run)
        else:
            run = 0
    return np.int16(best)


def rolling_sum_max(da, window):
    return da.rolling(time=window, min_periods=window).sum().max('time', skipna=True)


def save_index_dataset(index_dict, out_path, attrs):
    ds = xr.Dataset()
    for name, da in index_dict.items():
        ds[name] = da.astype('float32')
        ds[name].attrs['units'] = INDEX_UNITS.get(name, '')
    ds.attrs.update(attrs)
    encoding = {name: {'zlib': True, 'complevel': 4, 'dtype': 'float32'} for name in ds.data_vars}
    out_path.parent.mkdir(parents=True, exist_ok=True)
    ds.to_netcdf(out_path, encoding=encoding)
    ds.close()


def raw_index_path(model, scenario, year):
    return RAW_DIR / scenario / model / f'{model}_{scenario}_{year}_raw_etccdi.nc'


def era5_index_path(year):
    return ERA5_INDEX_DIR / f'ERA5_{year}_etccdi.nc'


def corrected_index_path(model, scenario, index_name):
    return CORR_DIR / scenario / model / f'{model}_{scenario}_{index_name}_annual_corrected.nc'

## Cell 3 - ETCCDI Index Calculators

In [ ]:
def compute_precip_indices(pr):
    wet = pr >= 1.0
    return {
        'PRCPTOT': pr.where(wet).sum('time', skipna=True),
        'RX1day': pr.max('time', skipna=True),
        'RX5day': rolling_sum_max(pr, 5),
        'SDII': pr.where(wet).sum('time', skipna=True) / wet.sum('time'),
        'R10mm': (pr >= 10.0).sum('time').astype('float32'),
        'R20mm': (pr >= 20.0).sum('time').astype('float32'),
        'CDD': xr.apply_ufunc(max_consecutive_true, pr < 1.0, input_core_dims=[['time']], vectorize=True, dask='parallelized', output_dtypes=[np.int16]),
        'CWD': xr.apply_ufunc(max_consecutive_true, pr >= 1.0, input_core_dims=[['time']], vectorize=True, dask='parallelized', output_dtypes=[np.int16]),
    }


def compute_temp_indices(tasmax, tasmin):
    return {
        'TXx': tasmax.max('time', skipna=True),
        'TXn': tasmax.min('time', skipna=True),
        'TNx': tasmin.max('time', skipna=True),
        'TNn': tasmin.min('time', skipna=True),
        'DTR': (tasmax - tasmin).mean('time', skipna=True),
        'SU': (tasmax > 25.0).sum('time').astype('float32'),
        'TR': (tasmin > 20.0).sum('time').astype('float32'),
        'FD': (tasmin < 0.0).sum('time').astype('float32'),
        'ID': (tasmax < 0.0).sum('time').astype('float32'),
    }

## Cell 4 - Compute ERA5 Annual Indices

In [ ]:
era5_saved = []
era5_failed = []

for year in range(OBS_PERIOD[0], OBS_PERIOD[1] + 1):
    out = era5_index_path(year)
    if out.exists() and out.stat().st_size > 0:
        print('[SKIP]', out.relative_to(ROOT))
        continue
    try:
        print('[ERA5]', year)
        pr = load_daily_file(era5_file('pr', year), 'pr')
        tx = load_daily_file(era5_file('tasmax', year), 'tasmax')
        tn = load_daily_file(era5_file('tasmin', year), 'tasmin')
        idx = {}
        idx.update(compute_precip_indices(pr))
        idx.update(compute_temp_indices(tx, tn))
        idx = {k: v.compute() for k, v in idx.items()}
        save_index_dataset(idx, out, {'source': 'ERA5-Land', 'year': year, 'note': 'Annual ETCCDI-style indices'})
        era5_saved.append(str(out))
        pr.close(); tx.close(); tn.close()
        gc.collect()
    except Exception as exc:
        msg = f'ERA5|{year}|{type(exc).__name__}: {exc}'
        print('[FAIL]', msg)
        era5_failed.append(msg)

(LOG_DIR / 'era5_index_failures.txt').write_text('\n'.join(era5_failed), encoding='utf-8')
print('ERA5 saved:', len(era5_saved), 'failed:', len(era5_failed))

## Cell 5 - Compute Raw CMIP6 Annual Indices

In [ ]:
raw_saved = []
raw_failed = []

for model in MODELS:
    variables = ['pr'] if model in PRECIP_ONLY_MODELS else ['pr', 'tasmax', 'tasmin']
    for scenario in SCENARIOS:
        for year in years_for(scenario):
            out = raw_index_path(model, scenario, year)
            if out.exists() and out.stat().st_size > 0:
                print('[SKIP]', out.relative_to(ROOT))
                continue
            try:
                print(f'[CMIP] {model} | {scenario} | {year}')
                idx = {}
                pr_file = cmip_file(model, scenario, 'pr', year)
                if is_structurally_readable(pr_file):
                    pr = load_daily_file(pr_file, 'pr')
                    idx.update(compute_precip_indices(pr))
                    pr.close()
                else:
                    raw_failed.append(f'SKIP_BAD_PR|{model}|{scenario}|{year}|{pr_file}')

                if {'tasmax', 'tasmin'}.issubset(set(variables)):
                    tx_file = cmip_file(model, scenario, 'tasmax', year)
                    tn_file = cmip_file(model, scenario, 'tasmin', year)
                    if is_structurally_readable(tx_file) and is_structurally_readable(tn_file):
                        tx = load_daily_file(tx_file, 'tasmax')
                        tn = load_daily_file(tn_file, 'tasmin')
                        idx.update(compute_temp_indices(tx, tn))
                        tx.close(); tn.close()
                    else:
                        raw_failed.append(f'SKIP_BAD_TEMP|{model}|{scenario}|{year}|tasmax_or_tasmin_bad')

                if idx:
                    idx = {k: v.compute() for k, v in idx.items()}
                    save_index_dataset(idx, out, {'source': 'NASA/GDDP-CMIP6', 'model': model, 'scenario': scenario, 'year': year, 'note': 'Raw annual ETCCDI-style indices before index-bias correction'})
                    raw_saved.append(str(out))
                    print('[OK]', out.name)
                gc.collect()
            except Exception as exc:
                msg = f'RAW_FAIL|{model}|{scenario}|{year}|{type(exc).__name__}: {exc}'
                print('[FAIL]', msg)
                raw_failed.append(msg)

(LOG_DIR / 'raw_cmip_index_failures_or_skips.txt').write_text('\n'.join(raw_failed), encoding='utf-8')
print('Raw CMIP saved:', len(raw_saved), 'failed/skipped:', len(raw_failed))

## Cell 6 - Index-Level Bias Correction Functions

In [ ]:
def force_numeric_index_array(da):
    da = da.squeeze(drop=True)
    da = standardise_xy(da)
    if np.issubdtype(da.dtype, np.timedelta64):
        da = da / np.timedelta64(1, 'D')
    elif not np.issubdtype(da.dtype, np.number):
        da = da.astype('float64')
    keep_coords = {c: da.coords[c] for c in ['lat', 'lon', 'year'] if c in da.coords}
    da = da.reset_coords(drop=True)
    if keep_coords:
        da = da.assign_coords(keep_coords)
    return da.astype('float32')


def get_index_stack_from_files(files, index_name):
    arrays = []
    years = []
    for f in files:
        with xr.open_dataset(f) as ds:
            if index_name not in ds.data_vars:
                continue
            m = re.search(r'_(\d{4})_', f.name)
            year = int(m.group(1)) if m else int(ds.attrs.get('year'))
            arrays.append(force_numeric_index_array(ds[index_name]).load())
            years.append(year)
    if not arrays:
        return None
    return xr.concat(arrays, dim=pd.Index(years, name='year')).sortby('year')


def bias_correct_index_series(raw_future, raw_hist, era5_hist, index_name):
    # Multiplicative for precipitation/count indices, additive for temperature indices.
    raw_hist_mean = raw_hist.mean('year', skipna=True)
    era5_mean = era5_hist.mean('year', skipna=True)
    if index_name in ['PRCPTOT', 'RX1day', 'RX5day', 'SDII', 'R10mm', 'R20mm', 'CDD', 'CWD']:
        factor = era5_mean / raw_hist_mean.where(np.abs(raw_hist_mean) > 1e-6)
        corrected = raw_future * factor
        corrected = corrected.where(corrected >= 0, 0)
    else:
        delta = era5_mean - raw_hist_mean
        corrected = raw_future + delta
    corrected.attrs.update({
        'bias_correction': 'annual index-level mean bias correction',
        'reference': 'ERA5-Land annual index mean, 1985-2014',
        'units': INDEX_UNITS.get(index_name, ''),
    })
    return corrected.astype('float32')


def raw_files_for(model, scenario):
    start, end = SCENARIOS[scenario]
    if TEST_YEARS is not None:
        start = max(start, TEST_YEARS[0])
        end = min(end, TEST_YEARS[1])
    return [raw_index_path(model, scenario, y) for y in range(start, end + 1) if raw_index_path(model, scenario, y).exists()]


def era5_index_files():
    return [era5_index_path(y) for y in range(BASELINE[0], BASELINE[1] + 1) if era5_index_path(y).exists()]


## Cell 7 - Bias-Correct Annual Index Series

In [ ]:
corr_saved = []
corr_failed = []
era5_files_all = era5_index_files()

for model in MODELS:
    available_indices = PRECIP_INDICES if model in PRECIP_ONLY_MODELS else ALL_INDICES
    hist_files = raw_files_for(model, 'historical')
    for index_name in available_indices:
        try:
            raw_hist = get_index_stack_from_files(hist_files, index_name)
            era5_hist = get_index_stack_from_files(era5_files_all, index_name)
            if raw_hist is None or era5_hist is None:
                corr_failed.append(f'MISSING_HIST|{model}|{index_name}')
                continue
            era5_hist = era5_hist.interp(lat=raw_hist.lat, lon=raw_hist.lon, method='nearest')

            for scenario in SCENARIOS:
                out = corrected_index_path(model, scenario, index_name)
                if (
                    (not OVERWRITE_CORRECTED)
                    and index_name not in FORCE_REPROCESS_CORRECTED_INDICES
                    and out.exists()
                    and out.stat().st_size > 0
                ):
                    print('[SKIP]', out.relative_to(ROOT))
                    continue
                raw_scen = get_index_stack_from_files(raw_files_for(model, scenario), index_name)
                if raw_scen is None:
                    corr_failed.append(f'MISSING_SCEN|{model}|{scenario}|{index_name}')
                    continue
                corrected = bias_correct_index_series(raw_scen, raw_hist, era5_hist, index_name)
                corrected = corrected.rename(index_name)
                corrected.attrs.update({'model': model, 'scenario': scenario, 'index': index_name})
                out.parent.mkdir(parents=True, exist_ok=True)
                corrected.to_dataset(name=index_name).to_netcdf(out, encoding={index_name: {'zlib': True, 'complevel': 4, 'dtype': 'float32'}})
                corr_saved.append(str(out))
                print('[OK]', out.relative_to(ROOT))
        except Exception as exc:
            msg = f'CORR_FAIL|{model}|{index_name}|{type(exc).__name__}: {exc}'
            print('[FAIL]', msg)
            corr_failed.append(msg)

(LOG_DIR / 'corrected_index_failures.txt').write_text('\n'.join(corr_failed), encoding='utf-8')
print('Corrected saved:', len(corr_saved), 'failed:', len(corr_failed))

## Cell 8 - Inventory

In [ ]:
rows = []
for f in sorted(CORR_DIR.rglob('*.nc')):
    parts = f.relative_to(CORR_DIR).parts
    scenario, model = parts[0], parts[1]
    idx = re.search(r'_([^_]+)_annual_corrected\.nc$', f.name).group(1)
    with xr.open_dataset(f) as ds:
        years = ds['year'].values if 'year' in ds.coords else []
    rows.append({
        'scenario': scenario,
        'model': model,
        'index': idx,
        'n_years': len(years),
        'first_year': int(years[0]) if len(years) else None,
        'last_year': int(years[-1]) if len(years) else None,
        'size_mb': round(f.stat().st_size / 1024 / 1024, 2),
        'path': str(f.relative_to(ROOT)),
    })

inventory = pd.DataFrame(rows)
inventory.to_csv(TABLE_DIR / 'fast_etccdi_corrected_index_inventory.csv', index=False)
print('Corrected index files:', len(inventory))
inventory.head()

## Cell 9 - Zone Mean Tables

In [ ]:
with xr.open_dataset(ZONES_FILE) as zds:
    zname = 'zone' if 'zone' in zds.data_vars else first_data_var(zds)
    zone_da = standardise_xy(zds[zname]).load()

ZONE_IDS = list(range(7))
ZONE_LABELS = {z: f'Z{z+1}' for z in ZONE_IDS}
zone_rows = []

for f in sorted(CORR_DIR.rglob('*.nc')):
    parts = f.relative_to(CORR_DIR).parts
    scenario, model = parts[0], parts[1]
    idx = re.search(r'_([^_]+)_annual_corrected\.nc$', f.name).group(1)
    with xr.open_dataset(f) as ds:
        da = force_numeric_index_array(ds[idx])
        z = zone_da.interp(lat=da.lat, lon=da.lon, method='nearest')
        for year in da.year.values:
            yda = da.sel(year=year)
            for zid in ZONE_IDS:
                zone_rows.append({
                    'scenario': scenario,
                    'model': model,
                    'index': idx,
                    'year': int(year),
                    'zone': ZONE_LABELS[zid],
                    'zone_id': zid,
                    'zone_mean': float(yda.where(z == zid).mean(['lat', 'lon'], skipna=True).values),
                    'units': INDEX_UNITS.get(idx, ''),
                })

zone_summary = pd.DataFrame(zone_rows)
zone_summary.to_csv(TABLE_DIR / 'fast_etccdi_zone_annual_means.csv', index=False)
print(zone_summary.shape)
zone_summary.head()

## Cell 10 - Period Change Tables

In [ ]:
PERIODS = {
    'baseline_1985_2014': (1985, 2014),
    'near_future_2021_2060': (2021, 2060),
    'far_future_2061_2100': (2061, 2100),
}

period_rows = []
for (scenario, model, idx, zone), sub in zone_summary.groupby(['scenario', 'model', 'index', 'zone']):
    for period, (start, end) in PERIODS.items():
        psub = sub[(sub.year >= start) & (sub.year <= end)]
        if psub.empty:
            continue
        period_rows.append({
            'scenario': scenario,
            'model': model,
            'index': idx,
            'zone': zone,
            'period': period,
            'mean_value': psub.zone_mean.mean(),
            'n_years': len(psub),
            'units': psub.units.iloc[0],
        })

period_df = pd.DataFrame(period_rows)
baseline = period_df[period_df.period == 'baseline_1985_2014'][['model', 'index', 'zone', 'mean_value']].rename(columns={'mean_value': 'baseline_mean'})
period_df = period_df.merge(baseline, on=['model', 'index', 'zone'], how='left')
period_df['absolute_change_vs_baseline'] = period_df['mean_value'] - period_df['baseline_mean']
period_df['percent_change_vs_baseline'] = 100 * period_df['absolute_change_vs_baseline'] / period_df['baseline_mean'].replace(0, np.nan)
period_df.to_csv(TABLE_DIR / 'fast_etccdi_period_changes_by_zone.csv', index=False)
period_df.head()

## Cell 11 - Diagnostic Figures

In [ ]:
mpl.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'axes.labelweight': 'bold',
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 130,
})

PLOT_INDICES = ['PRCPTOT', 'RX1day', 'CDD', 'TXx', 'TNn']
PANEL_LABELS = dict(zip(PLOT_INDICES, list('ABCDE')))
PLOT_TITLES = {
    'PRCPTOT': 'Annual wet-day precipitation',
    'RX1day': 'Maximum 1-day precipitation',
    'CDD': 'Consecutive dry days',
    'TXx': 'Annual maximum of daily Tmax',
    'TNn': 'Annual minimum of daily Tmin',
}
SCENARIO_LABELS = {'historical': 'Historical', 'ssp245': 'SSP2-4.5', 'ssp585': 'SSP5-8.5'}
SCENARIO_COLORS = {'historical': '#2F6B9A', 'ssp245': '#D98C19', 'ssp585': '#2C8C3C'}
SCENARIO_ORDER = ['historical', 'ssp245', 'ssp585']

def style_timeseries_axis(ax, idx, show_xlabel=True):
    ax.axvline(2015, color='0.25', lw=1.0, ls='--', alpha=0.75)
    ax.text(2015.8, 0.96, 'Scenario start', transform=ax.get_xaxis_transform(),
            ha='left', va='top', fontsize=8.5, fontweight='bold', color='0.25')
    ax.grid(True, axis='y', color='0.88', lw=0.8)
    ax.grid(True, axis='x', color='0.94', lw=0.5)
    ax.set_xlim(1984, 2101)
    ax.set_ylabel(f'{idx} ({INDEX_UNITS.get(idx, "")})', fontweight='bold')
    if show_xlabel:
        ax.set_xlabel('Year', fontweight='bold')
    for tick in ax.get_xticklabels() + ax.get_yticklabels():
        tick.set_fontweight('bold')


# Individual publication-ready figures.
for idx in PLOT_INDICES:
    sub = zone_summary[zone_summary['index'] == idx]
    if sub.empty:
        continue
    ens = sub.groupby(['scenario', 'year'], as_index=False)['zone_mean'].mean()
    fig, ax = plt.subplots(figsize=(10.5, 4.8))
    for scenario in SCENARIO_ORDER:
        g = ens[ens['scenario'] == scenario]
        if g.empty:
            continue
        ax.plot(g['year'], g['zone_mean'], label=SCENARIO_LABELS.get(scenario, scenario),
                color=SCENARIO_COLORS.get(scenario, '0.3'), lw=2.2)
    ax.set_title(f'{PANEL_LABELS[idx]}. {PLOT_TITLES[idx]}', loc='left', pad=8)
    style_timeseries_axis(ax, idx)
    ax.legend(frameon=False, ncol=3, loc='upper left')
    fig.tight_layout()
    fig.savefig(FIG_DIR / f'fast_etccdi_{idx}_scenario_timeseries.png', dpi=300, bbox_inches='tight')
    fig.savefig(FIG_DIR / f'fast_etccdi_{idx}_scenario_timeseries.pdf', bbox_inches='tight')
    plt.show()


# Combined multi-panel figure for manuscript/results section.
fig, axes = plt.subplots(3, 2, figsize=(13, 12), constrained_layout=True)
axes = axes.ravel()
for ax, idx in zip(axes, PLOT_INDICES):
    sub = zone_summary[zone_summary['index'] == idx]
    if sub.empty:
        ax.axis('off')
        continue
    ens = sub.groupby(['scenario', 'year'], as_index=False)['zone_mean'].mean()
    for scenario in SCENARIO_ORDER:
        g = ens[ens['scenario'] == scenario]
        if g.empty:
            continue
        ax.plot(g['year'], g['zone_mean'], label=SCENARIO_LABELS.get(scenario, scenario),
                color=SCENARIO_COLORS.get(scenario, '0.3'), lw=2.0)
    ax.set_title(f'{PANEL_LABELS[idx]}. {PLOT_TITLES[idx]}', loc='left', pad=7)
    style_timeseries_axis(ax, idx, show_xlabel=True)

axes[-1].axis('off')
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower right', bbox_to_anchor=(0.93, 0.08),
           frameon=False, ncol=1, title='Experiment')
fig.suptitle('Bias-corrected ETCCDI projections averaged across hydroclimatic zones',
             fontsize=15, fontweight='bold')
fig.savefig(FIG_DIR / 'fast_etccdi_main_indices_panel.png', dpi=300, bbox_inches='tight')
fig.savefig(FIG_DIR / 'fast_etccdi_main_indices_panel.pdf', bbox_inches='tight')
plt.show()

## Cell 12 - Save Summary

In [ ]:
summary = f'''# Fast ETCCDI Index Bias-Correction Summary

## Purpose

This workflow computes annual ETCCDI-style indices from raw daily CMIP6 and ERA5-Land, then bias-corrects the annual CMIP6 index series against ERA5 over 1985-2014. It is a fast alternative to daily-grid QDM.

## Outputs

- Raw CMIP6 annual indices: `output/etccdi_fast/raw_indices/`
- ERA5 annual indices: `output/etccdi_fast/era5_indices/`
- Corrected annual indices: `output/etccdi_fast/corrected_indices/`
- Zone annual means: `output/etccdi_fast/tables/fast_etccdi_zone_annual_means.csv`
- Period changes: `output/etccdi_fast/tables/fast_etccdi_period_changes_by_zone.csv`
- Inventory: `output/etccdi_fast/tables/fast_etccdi_corrected_index_inventory.csv`
- Figures: `output/etccdi_fast/figures/`
- Logs: `output/etccdi_fast/logs/`

## Method Note

Annual ETCCDI indices were first computed from daily fields. Bias correction was then applied to the annual index series using ERA5-Land as the 1985-2014 reference. This reduces computational cost while preserving the required annual index outputs for zone-level projections, trend analysis, and vegetation linkage assessment.
'''

(OUT_ROOT / 'PHASE_4_FAST_ETCCDI_SUMMARY.md').write_text(summary, encoding='utf-8')
print(summary)